[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/UrbanAccessAnalyzer/blob/main/examples/transit_level_of_service.ipynb)

# 🚌 Public Transport Level of Service

**How good is transit, street by street?** Every street gets a **level of service** (0-100) from the quality of the nearest stop (frequency, speed, mode) and how far it is on foot: the better the stop, the farther people walk to it.

*Everything in one call:* `transitlos.level_of_service_map(place)`.

In [ ]:
# Google Colab only: installs everything (all packages pinned to release v1.0.0). Does nothing locally.
import sys

if "google.colab" in sys.modules:
    !pip install -q "UrbanAccessAnalyzer[plot] @ git+https://github.com/CityScope/UrbanAccessAnalyzer.git@v1.0.0" "geohierarchy @ git+https://github.com/CityScope/geohierarchy.git@v1.0.0" "pycensus @ git+https://github.com/CityScope/pyCensus.git@v1.0.0" "pyGTFSHandler @ git+https://github.com/CityScope/pyGTFSHandler.git@v1.0.0" "transitlos @ git+https://github.com/CityScope/transitLOS.git@v1.0.0"
    # If an import fails afterwards: Runtime > Restart session, then run again.

In [ ]:
import sys
from datetime import date

from UrbanAccessAnalyzer import select_aoi, utils
from pyGTFSHandler.downloaders.mobility_database import download_gtfs_for_place
from transitlos import (compute_level_of_service, compute_stop_scores, download_and_prepare_stops,
                        level_of_service_results, prepare_street_network, top_stops)

# 1 Area of interest

Write a **place name** (its boundary is drawn on the map) or **draw your own polygon or rectangle** with the tools on the left. If you draw nothing, the place is used.

In [ ]:
# USER INPUT
place = "Cambridge, Massachusetts, USA"  # a place name; or draw your own area on the map below (then this only names the output folder)
folder = utils.output_folder(place)  # results are saved here (the folder is created)
cache_dir = "cache"  # downloads (place, streets, population) are kept here, next to the notebook, and reused when you rerun

selector = select_aoi(place, cache_dir=cache_dir)  # searches the place
selector.map  # the place, dashed blue. Draw a polygon or rectangle here to use your own area instead

In [ ]:
aoi = selector.aoi  # the area used from here on: what you drew, or else the place

# Settings

In [ ]:
# USER INPUT
region = "north_america"  # "europe", "north_america", "global_south" or "global"
service_day = None  # e.g. date(2026, 8, 21). None: a typical day is detected automatically (slower)
max_walk_distance = 2000  # meters; streets farther than this from any stop score 0
h3_resolution = 10  # hexagon size of the results: 10 is ~0.015 km2, 9 is ~0.1 km2
population = "worldpop"  # "worldpop": 100 m grid, anywhere in the world | "census": the country's own census (an error if not implemented)
empty_cells = "nearest_value"  # hexagons with people but no street: "nearest_value" (copy the nearest hexagon's value) | "relocate_weight" (move their people to the nearest one, within 300 m) | "ignore" (no score)

# 2 Data download

The study area, the GTFS feeds ([Mobility Database](https://mobilitydatabase.org/): a free token, set as the `MOBILITY_DATABASE_REFRESH_TOKEN` environment variable or pasted when asked; or your own with `feed_source`), the walking street network and the stops of a service day. Downloads are kept in the `cache` folder and reused when you rerun.

In [ ]:
feed_source = download_gtfs_for_place(place, f"{cache_dir}/gtfs/{utils.sanitize_filename(place)}")  # or: ["my_feed.zip", ...]
network = prepare_street_network(aoi, cache_dir=f"{cache_dir}/networks", pbf_path=f"{cache_dir}/pbf/region.osm.pbf", cache_by_aoi=True)
stops = download_and_prepare_stops(feed_source, aoi=utils.buffer_gdf(aoi, 500),  # stops just outside the area also serve it
                                   date_range=(service_day, service_day) if service_day else None)
print(f"{len(stops)} stops")

# 3 Scoring the stops

Each stop gets a quality score (0 to 1) from its mode, how frequent it is (`headway_minutes`) and how fast (`avg_speed_kmh`). The best stops:

In [ ]:
stops = compute_stop_scores(stops, region=region)
top_stops(stops)  # the best stops

# 4 Isochrones

From every stop we walk along the streets: each street gets the best stop quality, decayed by the walking distance (0 beyond `max_walk_distance`).

In [ ]:
edges = compute_level_of_service(aoi, network=network, stops=stops, region=region, max_walk_distance_m=max_walk_distance)
edges.to_file(f"{folder}/transit_streets.gpkg")
edges["level_of_service"].describe().round(1)

# 5 Results

In [ ]:
hexes, stats, m = level_of_service_results(edges, aoi, folder, h3_resolution, population, "transit_map.html", empty_cells=empty_cells, cache_dir=cache_dir)
m

**Who is well served?** People living in each band of service:

In [ ]:
stats  # also saved as stats.csv

In [ ]:
if "google.colab" in sys.modules:
    !zip -r /content/output.zip output  # then download it from the file browser

For the weighted median, Gini coefficient and breakdowns by group, see `transitlos.equity_statistics` and the `equity_analysis_walkthrough.ipynb` notebook of the transitLOS repository.